# Checkpointers (Part 2) — SQLite-Backed Persistent Memory

This notebook extends the [Agents](../../LangChain%20Agents%20%26%20Tools/Labs/Agents.ipynb) example by replacing the RAM-only `InMemorySaver` with a **SQLite-backed checkpointer** that persists state to a real database file.

## Key concepts

- **`SqliteSaver`** – Stores agent checkpoints in a SQLite database file on disk. Data survives after the Python process ends, enabling multi-session memory.
- **SQLite** – A lightweight, file-based SQL database built into Python's standard library. No separate server needed.
- **`pandas` DataFrame** – Used here to display checkpoint tables in a readable, structured format inside the notebook.
- **Two checkpoint tables** – LangGraph creates two tables inside the database:
  - `checkpoints` – One row per checkpoint (agent state snapshot at a particular step).
  - `writes` – One row per state update written during a step (more granular than checkpoints).

## What's different from Part 1?

| Part 1 (`InMemorySaver`) | Part 2 (`SqliteSaver`) |
|---|---|
| Data lives in RAM | Data lives in a `.db` file |
| Lost when notebook restarts | Survives restarts |
| No SQL inspection needed | Can query with pandas/SQL |
| Best for: quick demos | Best for: real apps with persistence |

In [ ]:
# Install additional packages:
# - langgraph-checkpoint-sqlite: the SQLite checkpointer adapter for LangGraph
# - pandas: for reading and displaying database tables as DataFrames
!pip install -q langchain langchain-openai langgraph-checkpoint-sqlite pandas

In [ ]:
import sqlite3   # Python's built-in SQLite library — no installation needed
import pandas as pd  # For displaying database tables as readable DataFrames

from google.colab import userdata
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver  # The SQLite checkpointer
from pydantic import SecretStr
from typing import List

# Securely load the OpenAI API key.
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper to print the conversation messages in a human-readable format.
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
# Open a connection to the SQLite database file.
# The file will be created if it doesn't exist yet.
# check_same_thread=False is required because LangGraph may access the DB from different threads.
sqlite_connection = sqlite3.connect("/content/checkpointer.db", check_same_thread=False)

# Create the SQLite-backed checkpointer, passing in the open database connection.
# LangGraph will automatically create the required tables on first use.
checkpointer = SqliteSaver(sqlite_connection)

In [ ]:
# Helper function to read and return ALL tables from the SQLite database as pandas DataFrames.
# We use this to inspect the raw checkpoint data that LangGraph writes to the DB.
def explore_database():
    # Query the SQLite master table to get a list of all table names.
    tables_df = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table';", sqlite_connection)

    result = {}
    # Loop through each table name and load its entire contents into a DataFrame.
    for table_name in tables_df["name"]:
        result[table_name] = pd.read_sql_query(f"SELECT * FROM \"{table_name}\"", sqlite_connection)

    return result  # Returns a dict like: { "checkpoints": DataFrame, "writes": DataFrame }

In [ ]:
# --- Define the same two tools as in Part 1 ---
# (Identical tools are used so you can focus on the checkpointer difference.)

@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """
    # Hardcoded tour stop data — simulates a real tour calendar API.
    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    # Hardcoded drive time estimates between Bulgarian city pairs.
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")

In [ ]:
# Build the agent — identical to Part 1, but using SqliteSaver instead of InMemorySaver.
# From the agent's perspective, the API is exactly the same.
# The only difference is WHERE the checkpoints are stored.
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key),
    tools=[lookup_tour_stop, estimate_drive_time],
    system_prompt="You are a practical live-music concierge. Use tools when they help you give a precise answer.",
    checkpointer=checkpointer  # SQLite-backed checkpointer
)

In [ ]:
# Thread configuration — same pattern as Part 1.
# thread_id "1" identifies this specific conversation session.
config = {
    "configurable": {
        "thread_id": "1"
    }
}

In [ ]:
# Send the first user message and run the agent.
# The agent will use tools, then save the resulting state to the SQLite database.
plan_concert_trip = agent.invoke(
    input={
      "messages": HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time.")
    },
    config=config
)

In [ ]:
# Display the first conversation turn.
print_conversation(plan_concert_trip["messages"])

In [ ]:
# Capture a snapshot of the database AFTER the first agent invocation.
# We'll compare this with a second snapshot later to see what changed.
first_exploration = explore_database()

In [ ]:
# Display the `checkpoints` table — one row per state snapshot.
# Each row represents the complete agent state at a specific point in the execution.
first_exploration["checkpoints"]

In [ ]:
# Display the `writes` table — one row per individual state update (more granular than checkpoints).
# This shows every value written to the state channels during execution.
first_exploration["writes"]

In [ ]:
# Send the follow-up message using the SAME thread_id.
# The agent will load its state from the SQLite DB and "remember" the previous turn.
test_agent_memory = agent.invoke(
    input={
        "messages": [HumanMessage("What do you remember about me?")]
    },
    config=config
)

In [ ]:
# Display the conversation for the second turn.
print_conversation(test_agent_memory["messages"])

In [ ]:
# Capture the database state AFTER the second invocation.
# Compare with first_exploration to see how many new checkpoint rows were added.
second_exploration = explore_database()

In [ ]:
# Compare the checkpoints table after the second invocation.
# There should be more rows than in first_exploration["checkpoints"].
second_exploration["checkpoints"]

In [ ]:
# Compare the writes table after the second invocation.
# More rows means more state updates were written during the second agent run.
second_exploration["writes"]